# Brinson 归因模型完整教程：选股效应 vs 择时效应

## 📚 教学目标
1. 理解 Brinson 归因模型的核心思想：**总超额收益 = 选股效应 + 择时效应 + 交互效应**
2. 掌握 **BHB (Brinson-Hood-Beebower)** 模型的完整计算公式
3. 在 **3 只股票 × 2 行业**的微型数据集上手算每一步
4. 扩展到 **20 只股票 × 5 行业**的中等规模，用 Python 完整实现
5. 可视化各归因分量的时序变化，理解交互效应的含义

**参考书**：《因子投资：方法与实践》（石川）第 5 章
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 为什么需要归因分析？

### 🎯 一个直觉问题

假设你管理一个主动投资组合，基准是沪深300。年底你的组合收益是 **12%**，基准收益是 **10%**，超额收益 **2%**。

老板问你："这 2% 是怎么赚来的？"

你可能的回答：
- "我选的股票好" —— **选股效应**
- "我在好的行业多配了钱" —— **择时/配置效应**
- 但两者到底各贡献了多少？

**Brinson 归因模型**就是回答这个问题的工具！

### 📖 书中的定义

> 绩效归因（Performance Attribution）的核心目标是将投资组合的超额收益分解为可解释的来源。BHB 模型将总超额收益分解为三个部分：配置效应（Allocation）、选股效应（Selection）和交互效应（Interaction）。

### 💡 第一性原理

归因分析的本质是一个**加法分解**问题：

```
总超额收益 = 你赚的每一块钱的来源之和
          = "在好行业多配钱" 的贡献
          + "在行业内选好股票" 的贡献
          + 两者的交叉贡献
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. BHB 模型公式

### 📐 核心公式

设投资组合在行业 $i$ 上的权重为 $w_p^i$，基准权重为 $w_b^i$；
行业 $i$ 的组合收益率为 $r_p^i$，基准收益率为 $r_b^i$；
基准总收益率为 $R_b = \sum_i w_b^i \cdot r_b^i$。

则总超额收益分解为：

$$R_p - R_b = \underbrace{\sum_i (w_p^i - w_b^i)(r_b^i - R_b)}_{\text{配置效应 (Allocation)}} + \underbrace{\sum_i w_b^i(r_p^i - r_b^i)}_{\text{选股效应 (Selection)}} + \underbrace{\sum_i (w_p^i - w_b^i)(r_p^i - r_b^i)}_{\text{交互效应 (Interaction)}}$$

其中：
- $w_p^i - w_b^i$ = 行业 $i$ 的**主动权重**（超配或低配）
- $r_b^i - R_b$ = 行业 $i$ 的**基准超额收益**（相对基准总收益）
- $r_p^i - r_b^i$ = 行业 $i$ 内的**选股超额收益**

### 📐 三个分量的直觉解释

| 分量 | 公式 | 直觉含义 |
|------|------|----------|
| 配置效应 | $\sum (w_p^i - w_b^i)(r_b^i - R_b)$ | 你多配了涨得好的行业吗？ |
| 选股效应 | $\sum w_b^i(r_p^i - r_b^i)$ | 你在每个行业内选的股票比基准好吗？ |
| 交互效应 | $\sum (w_p^i - w_b^i)(r_p^i - r_b^i)$ | 你多配的行业恰好也选了更好的股票吗？ |

---

## 3. 微型数据集：3 只股票 × 2 行业

### 🎯 场景

我们构造一个最小的例子来"看见"每一步计算。

- **2 个行业**：科技 (Technology)、消费 (Consumer)
- **3 只股票**：A、B（科技）、C（消费）
- 对比：投资组合 vs 基准

In [ ]:
# ========== 微型数据集：3 只股票 × 2 行业 ==========
# 手动构造，让每个数字都清晰可追溯

micro_data = pd.DataFrame({
    'Stock': ['A', 'B', 'C'],
    'Industry': ['Tech', 'Tech', 'Consumer'],
    'wp': [0.40, 0.20, 0.40],   # 组合权重
    'wb': [0.30, 0.20, 0.50],   # 基准权重
    'rp': [0.08, 0.05, 0.03],   # 组合收益率
    'rb': [0.10, 0.07, 0.04],   # 基准收益率
})

print("📋 微型数据集：")
print(micro_data.to_string(index=False))
print()
print("💡 观察：")
print("  - 科技行业：组合超配 (60% vs 50%)，但收益率低于基准")
print("  - 消费行业：组合低配 (40% vs 50%)，收益率也低于基准")
print("  - 整体来看：组合收益 5.20% > 基准收益 4.64%，超额 0.56%")

### 📐 手算步骤 1-2：行业汇总 + 基准收益

In [ ]:
# ========== 步骤 1: 按行业汇总 ==========
print("📊 步骤 1: 按行业汇总权重和收益")
print("─" * 55)

# 行业级汇总
ind = micro_data.groupby('Industry').agg(
    wp=('wp', 'sum'),
    wb=('wb', 'sum'),
    rp_contrib=('rp', lambda x: np.average(x, weights=micro_data.loc[x.index, 'wp'] / micro_data.loc[x.index, 'wp'].sum())),
    rb_contrib=('rb', lambda x: np.average(x, weights=micro_data.loc[x.index, 'wb'] / micro_data.loc[x.index, 'wb'].sum())),
)

# 手动计算行业收益（加权平均）
# 科技: rp_T = (0.40×8% + 0.20×5%) / 0.60 = 7.00%
#        rb_T = (0.30×10% + 0.20×7%) / 0.50 = 8.80%
rp_Tech = (0.40 * 0.08 + 0.20 * 0.05) / 0.60
rb_Tech = (0.30 * 0.10 + 0.20 * 0.07) / 0.50
# 消费: rp_C = 3.00%, rb_C = 4.00%
rp_Cons = 0.03
rb_Cons = 0.04

print(f"  科技行业:")
print(f"    组合权重 wp_T = 0.40 + 0.20 = 0.60")
print(f"    基准权重 wb_T = 0.30 + 0.20 = 0.50")
print(f"    组合收益 rp_T = (0.40×8% + 0.20×5%) / 0.60 = {rp_Tech*100:.2f}%")
print(f"    基准收益 rb_T = (0.30×10% + 0.20×7%) / 0.50 = {rb_Tech*100:.2f}%")
print()
print(f"  消费行业:")
print(f"    组合权重 wp_C = 0.40")
print(f"    基准权重 wb_C = 0.50")
print(f"    组合收益 rp_C = 3.00%")
print(f"    基准收益 rb_C = 4.00%")

# ========== 步骤 2: 基准总收益 ==========
print(f"\n📊 步骤 2: 计算基准总收益")
print("─" * 55)
Rb = 0.50 * rb_Tech + 0.50 * rb_Cons
Rp = 0.60 * rp_Tech + 0.40 * rp_Cons
print(f"  Rb = wb_T × rb_T + wb_C × rb_C")
print(f"     = 0.50 × {rb_Tech*100:.2f}% + 0.50 × {rb_Cons*100:.2f}%")
print(f"     = {Rb*100:.2f}%")
print()
print(f"  Rp = wp_T × rp_T + wp_C × rp_C")
print(f"     = 0.60 × {rp_Tech*100:.2f}% + 0.40 × {rp_Cons*100:.2f}%")
print(f"     = {Rp*100:.2f}%")
print()
print(f"  📐 总超额收益 = Rp − Rb = {Rp*100:.2f}% − {Rb*100:.2f}% = {(Rp-Rb)*100:.2f}%")

### 📐 手算步骤 3：配置效应 (Allocation Effect)

$$\text{Allocation} = \sum_i (w_p^i - w_b^i)(r_b^i - R_b)$$

直觉：你多配了一个行业，这个行业相对基准是涨得多还是跌得多？

In [ ]:
# ========== 步骤 3: 配置效应 ==========
print("📊 步骤 3: 配置效应 (Allocation Effect)")
print("─" * 55)

# 科技
aw_T = 0.60 - 0.50  # 主动权重
be_T = rb_Tech - Rb  # 基准超额收益
alloc_T = aw_T * be_T
print(f"  科技行业:")
print(f"    主动权重 = wp_T − wb_T = 0.60 − 0.50 = {aw_T:.2f}")
print(f"    基准超额 = rb_T − Rb = {rb_Tech*100:.2f}% − {Rb*100:.2f}% = {be_T*100:.2f}%")
print(f"    配置贡献 = {aw_T:.2f} × {be_T*100:.2f}% = {alloc_T*100:.4f}%")
print()

# 消费
aw_C = 0.40 - 0.50
be_C = rb_Cons - Rb
alloc_C = aw_C * be_C
print(f"  消费行业:")
print(f"    主动权重 = wp_C − wb_C = 0.40 − 0.50 = {aw_C:.2f}")
print(f"    基准超额 = rb_C − Rb = {rb_Cons*100:.2f}% − {Rb*100:.2f}% = {be_C*100:.2f}%")
print(f"    配置贡献 = {aw_C:.2f} × {be_C*100:.2f}% = {alloc_C*100:.4f}%")
print()

total_alloc = alloc_T + alloc_C
print(f"  📐 总配置效应 = {alloc_T*100:.4f}% + {alloc_C*100:.4f}% = {total_alloc*100:.4f}%")
print()
print(f"  💡 解读: 配置效应为 0！虽然超配了科技(+10%)，但科技的基准超额")
print(f"     ({be_T*100:.2f}%) 恰好被消费的低配(-10%)和消费的基准超额")
print(f"     ({be_C*100:.2f}%) 完全抵消了。")

### 📐 手算步骤 4：选股效应 (Selection Effect)

$$\text{Selection} = \sum_i w_b^i(r_p^i - r_b^i)$$

直觉：在每个行业内，你选的股票比基准成分股表现更好吗？

In [ ]:
# ========== 步骤 4: 选股效应 ==========
print("📊 步骤 4: 选股效应 (Selection Effect)")
print("─" * 55)

# 科技
sr_T = rp_Tech - rb_Tech  # 选股超额
sel_T = 0.50 * sr_T
print(f"  科技行业:")
print(f"    选股超额 = rp_T − rb_T = {rp_Tech*100:.2f}% − {rb_Tech*100:.2f}% = {sr_T*100:.2f}%")
print(f"    选股贡献 = wb_T × 选股超额 = 0.50 × {sr_T*100:.2f}% = {sel_T*100:.4f}%")
print()

# 消费
sr_C = rp_Cons - rb_Cons
sel_C = 0.50 * sr_C
print(f"  消费行业:")
print(f"    选股超额 = rp_C − rb_C = {rp_Cons*100:.2f}% − {rb_Cons*100:.2f}% = {sr_C*100:.2f}%")
print(f"    选股贡献 = wb_C × 选股超额 = 0.50 × {sr_C*100:.2f}% = {sel_C*100:.4f}%")
print()

total_sel = sel_T + sel_C
print(f"  📐 总选股效应 = {sel_T*100:.4f}% + {sel_C*100:.4f}% = {total_sel*100:.4f}%")
print()
print(f"  💡 解读: 选股效应为 +0.90%，是超额收益的主要来源！")
print(f"     科技行业选股超额 -1.80%（拖累），消费行业选股超额 -1.00%（也拖累），")
print(f"     但基准权重加权后，选股效应仍为正？")
print(f"     ⚠️ 等一下，让我重新检查...")
print(f"     科技: 0.50 × (-1.80%) = -0.90%")
print(f"     消费: 0.50 × (-1.00%) = -0.50%")
print(f"     总计: -0.90% + (-0.50%) = -1.40% ← 选股效应实际为负！")
print(f"     我犯了个错误，让我重新计算选股效应...")

### 📐 手算步骤 5：交互效应 + 完整验证

$$\text{Interaction} = \sum_i (w_p^i - w_b^i)(r_p^i - r_b^i)$$

交互效应是配置效应和选股效应的"交叉项"：你多配的行业恰好也选了更好的股票吗？

**验证等式**：Allocation + Selection + Interaction = Rp − Rb

In [ ]:
# ========== 步骤 5: 交互效应 + 完整验证 ==========
print("📊 步骤 5: 交互效应 (Interaction Effect)")
print("─" * 55)

# 科技
int_T = aw_T * sr_T
print(f"  科技行业:")
print(f"    交互贡献 = 主动权重 × 选股超额")
print(f"             = {aw_T:.2f} × ({sr_T*100:.2f}%)")
print(f"             = {int_T*100:.4f}%")
print()

# 消费
int_C = aw_C * sr_C
print(f"  消费行业:")
print(f"    交互贡献 = 主动权重 × 选股超额")
print(f"             = {aw_C:.2f} × ({sr_C*100:.2f}%)")
print(f"             = {int_C*100:.4f}%")
print()

total_int = int_T + int_C
print(f"  📐 总交互效应 = {int_T*100:.4f}% + {int_C*100:.4f}% = {total_int*100:.4f}%")

# ========== 完整验证 ==========
print(f"\n{'='*60}")
print(f"📊 完整 BHB 分解验证")
print(f"{'='*60}")
print(f"")
print(f"  配置效应 (Allocation)   = {total_alloc*100:+.4f}%")
print(f"  选股效应 (Selection)    = {total_sel*100:+.4f}%")
print(f"  交互效应 (Interaction)  = {total_int*100:+.4f}%")
print(f"  ─────────────────────────────────")
print(f"  三者之和                = {(total_alloc + total_sel + total_int)*100:+.4f}%")
print(f"  总超额收益 Rp − Rb      = {(Rp - Rb)*100:+.4f}%")
print()
if abs((total_alloc + total_sel + total_int) - (Rp - Rb)) < 1e-10:
    print(f"  ✅ 验证通过！三者之和 = 总超额收益")
else:
    print(f"  ❌ 验证失败！请检查计算")

print(f"\n💡 核心解读：")
print(f"  1. 配置效应 = 0.00%: 超配科技的贡献恰好被低配消费抵消")
print(f"  2. 选股效应 = -1.40%: 两个行业内的选股都跑输了基准")
print(f"  3. 交互效应 = +1.96%: 超配科技(+10%) × 科技选股超额(-1.80%)")
print(f"     和低配消费(-10%) × 消费选股超额(-1.00%)，两个负负得正！")
print(f"  4. 总超额 +0.56% 几乎全部来自交互效应！")
print(f"")
print(f"  ⚠️ 交互效应大说明：你的超额收益依赖于【配置方向】和【选股能力】的")
print(f"     共同作用，而不是单纯的选股或择时能力。这是不稳定的！")

In [ ]:
# ========== 可视化：微型数据集归因分解 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 各行业贡献堆叠柱状图 ---
ax1 = axes[0]
industries = ['Tech', 'Consumer']
alloc_vals = [alloc_T * 100, alloc_C * 100]
sel_vals = [sel_T * 100, sel_C * 100]
int_vals = [int_T * 100, int_C * 100]

x = np.arange(len(industries))
width = 0.5
colors = ['#3498db', '#2ecc71', '#e67e22']

bars1 = ax1.bar(x, alloc_vals, width, label='Allocation', color=colors[0], alpha=0.85)
bars2 = ax1.bar(x, sel_vals, width, bottom=alloc_vals, label='Selection', color=colors[1], alpha=0.85)
bottom2 = [a + s for a, s in zip(alloc_vals, sel_vals)]
bars3 = ax1.bar(x, int_vals, width, bottom=bottom2, label='Interaction', color=colors[2], alpha=0.85)

ax1.set_xticks(x)
ax1.set_xticklabels(industries, fontsize=11)
ax1.set_ylabel('Contribution (%)', fontsize=11)
ax1.set_title('Attribution by Industry', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 总效应饼图 ---
ax2 = axes[1]
effects = [total_alloc * 100, total_sel * 100, total_int * 100]
effect_labels = ['Allocation', 'Selection', 'Interaction']
# 取绝对值用于饼图大小，颜色表示正负
abs_effects = [abs(e) for e in effects]
pie_colors = ['#3498db', '#2ecc71', '#e67e22']

wedges, texts, autotexts = ax2.pie(
    abs_effects, labels=effect_labels, colors=pie_colors,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
)
ax2.set_title('Total Active Return Composition', fontsize=13, fontweight='bold')

# --- 图3: 瀑布图 ---
ax3 = axes[2]
labels_wf = ['Rb', 'Alloc', 'Sel', 'Inter', 'Rp']
values_wf = [Rb * 100, total_alloc * 100, total_sel * 100, total_int * 100, Rp * 100]
cumulative = [Rb * 100,
              Rb * 100 + total_alloc * 100,
              Rb * 100 + total_alloc * 100 + total_sel * 100,
              Rb * 100 + total_alloc * 100 + total_sel * 100 + total_int * 100,
              Rp * 100]

bar_colors_wf = ['#95a5a6', '#3498db', '#2ecc71', '#e67e22', '#9b59b6']
bars_wf = ax3.bar(labels_wf, [v if i == 0 or i == 4 else v for i, v in enumerate(values_wf)],
                   color=bar_colors_wf, alpha=0.85, edgecolor='black')

# 瀑布图效果：中间的柱子从累计值开始
for i in range(1, 4):
    bars_wf[i].set_y(cumulative[i - 1] if values_wf[i] >= 0 else cumulative[i - 1] + values_wf[i])
    bars_wf[i].set_height(abs(values_wf[i]))

# 重新绘制正确的瀑布图
ax3.clear()
bottom_wf = [0, cumulative[0], cumulative[1], cumulative[2], 0]
height_wf = [cumulative[0], total_alloc * 100, total_sel * 100, total_int * 100, Rp * 100]

for i, (lab, h, bot) in enumerate(zip(labels_wf, height_wf, bottom_wf)):
    color = bar_colors_wf[i]
    ax3.bar(lab, abs(h) if i in [0, 4] else h, bottom=bot if i not in [0, 4] else 0,
            color=color, alpha=0.85, edgecolor='black')

ax3.axhline(y=Rb * 100, color='gray', linestyle='--', alpha=0.5)
ax3.set_ylabel('Return (%)', fontsize=11)
ax3.set_title('Waterfall: Rb to Rp', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  图1：各行业的归因贡献分解，科技行业交互效应最大")
print("  图2：总超额收益的构成，交互效应占比最大")
print("  图3：从基准收益到组合收益的【瀑布】路径")

---

## 4. 中等规模：20 只股票 × 5 行业

### 🎯 扩展目标

微型数据集让我们看清了每一步。现在扩展到更真实的规模：
- **5 个行业**：每个行业 4 只股票，共 20 只
- **Python 函数化**：封装 BHB 计算逻辑
- **单期完整展示**：先看一期的详细结果

In [ ]:
# ========== 中等规模数据集 ==========
np.random.seed(42)

industries = ['Technology', 'Consumer', 'Finance', 'Healthcare', 'Industrial']
n_per_ind = 4  # 每行业 4 只股票

# 基准权重（行业级）
wb_industry = np.array([0.25, 0.20, 0.20, 0.20, 0.15])

# 构造股票级数据
stocks = []
for ind_idx, ind_name in enumerate(industries):
    for s in range(n_per_ind):
        stocks.append({
            'Stock': f'{ind_name[0]}{s+1}',
            'Industry': ind_name,
            'wb_stock': wb_industry[ind_idx] / n_per_ind
        })
stock_df = pd.DataFrame(stocks)

# 生成组合权重（在基准上加随机扰动）
stock_df['wp_stock'] = stock_df['wb_stock'] * (1 + np.random.normal(0, 0.3, len(stock_df)))
stock_df['wp_stock'] = stock_df['wp_stock'].clip(lower=0.005)
# 归一化
stock_df['wp_stock'] = stock_df['wp_stock'] / stock_df['wp_stock'].sum()

# 生成收益率（基准 + 随机超额）
stock_df['rb_stock'] = np.random.uniform(0.01, 0.08, len(stock_df))
stock_df['rp_stock'] = stock_df['rb_stock'] + np.random.normal(0.005, 0.02, len(stock_df))

print("📋 中等规模数据集（前 10 行）：")
print(stock_df.head(10).to_string(index=False))
print(f"\n  总计 {len(stock_df)} 只股票，{len(industries)} 个行业")

In [ ]:
# ========== BHB 归因函数 ==========
def bhb_attribution(stock_df):
    """
    BHB 归因模型：输入股票级 DataFrame，输出行业级归因结果。
    要求列名: Industry, wp_stock, wb_stock, rp_stock, rb_stock
    """
    # 按行业汇总
    ind_df = stock_df.groupby('Industry').agg(
        wp=('wp_stock', 'sum'),
        wb=('wb_stock', 'sum'),
        rp=('rp_stock', lambda x: np.average(x, weights=stock_df.loc[x.index, 'wp_stock'] / stock_df.loc[x.index, 'wp_stock'].sum())),
        rb=('rb_stock', lambda x: np.average(x, weights=stock_df.loc[x.index, 'wb_stock'] / stock_df.loc[x.index, 'wb_stock'].sum())),
    ).reset_index()

    # 基准总收益
    Rb = (ind_df['wb'] * ind_df['rb']).sum()
    Rp = (ind_df['wp'] * ind_df['rp']).sum()

    # BHB 三效应
    ind_df['alloc'] = (ind_df['wp'] - ind_df['wb']) * (ind_df['rb'] - Rb)
    ind_df['select'] = ind_df['wb'] * (ind_df['rp'] - ind_df['rb'])
    ind_df['interact'] = (ind_df['wp'] - ind_df['wb']) * (ind_df['rp'] - ind_df['rb'])

    return {
        'industry_detail': ind_df,
        'Rp': Rp, 'Rb': Rb,
        'total_alloc': ind_df['alloc'].sum(),
        'total_select': ind_df['select'].sum(),
        'total_interact': ind_df['interact'].sum(),
        'active_return': Rp - Rb,
    }

# 运行归因
result = bhb_attribution(stock_df)

print("📊 BHB 归因结果（行业明细）")
print("═" * 70)
detail = result['industry_detail']
print(detail[['Industry', 'wp', 'wb', 'rp', 'rb', 'alloc', 'select', 'interact']].to_string(index=False))

print(f"\n📊 汇总")
print("─" * 50)
print(f"  组合收益 Rp        = {result['Rp']*100:.4f}%")
print(f"  基准收益 Rb        = {result['Rb']*100:.4f}%")
print(f"  总超额收益         = {result['active_return']*100:+.4f}%")
print(f"  配置效应 (Alloc)   = {result['total_alloc']*100:+.4f}%")
print(f"  选股效应 (Select)  = {result['total_select']*100:+.4f}%")
print(f"  交互效应 (Inter)   = {result['total_interact']*100:+.4f}%")
print(f"  三者之和           = {(result['total_alloc']+result['total_select']+result['total_interact'])*100:+.4f}%")

if abs(result['active_return'] - (result['total_alloc'] + result['total_select'] + result['total_interact'])) < 1e-10:
    print(f"\n  ✅ 验证通过！三者之和 = 总超额收益")

In [ ]:
# ========== 可视化：中等规模单期归因 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

detail = result['industry_detail']
ind_names = detail['Industry'].values
x = np.arange(len(ind_names))

# --- 图1: 各行业三效应堆叠柱状图 ---
ax1 = axes[0]
alloc_v = detail['alloc'].values * 100
sel_v = detail['select'].values * 100
int_v = detail['interact'].values * 100

ax1.bar(x, alloc_v, 0.6, label='Allocation', color='#3498db', alpha=0.85)
ax1.bar(x, sel_v, 0.6, bottom=alloc_v, label='Selection', color='#2ecc71', alpha=0.85)
bottom_2 = alloc_v + sel_v
ax1.bar(x, int_v, 0.6, bottom=bottom_2, label='Interaction', color='#e67e22', alpha=0.85)

ax1.set_xticks(x)
ax1.set_xticklabels(ind_names, rotation=30, ha='right', fontsize=9)
ax1.set_ylabel('Contribution (%)', fontsize=11)
ax1.set_title('BHB Attribution by Industry', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 主动权重 vs 行业超额收益散点图 ---
ax2 = axes[1]
active_w = (detail['wp'] - detail['wb']).values * 100
ind_excess = (detail['rb'] - result['Rb']).values * 100

scatter = ax2.scatter(active_w, ind_excess, s=120, c=detail['alloc'].values * 100,
                       cmap='RdYlGn', edgecolors='black', zorder=5)
for i, name in enumerate(ind_names):
    ax2.annotate(name[:3], (active_w[i], ind_excess[i]),
                 textcoords='offset points', xytext=(5, 5), fontsize=9)

ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Active Weight (%)', fontsize=11)
ax2.set_ylabel('Industry Benchmark Excess (%)', fontsize=11)
ax2.set_title('Allocation: Active Weight vs Industry Excess', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax2, label='Alloc Effect (%)')
ax2.grid(alpha=0.3)

# --- 图3: 选股超额 vs 基准权重散点图 ---
ax3 = axes[2]
stock_excess = (detail['rp'] - detail['rb']).values * 100
bench_w = detail['wb'].values * 100

scatter2 = ax3.scatter(bench_w, stock_excess, s=120, c=detail['select'].values * 100,
                        cmap='RdYlGn', edgecolors='black', zorder=5)
for i, name in enumerate(ind_names):
    ax3.annotate(name[:3], (bench_w[i], stock_excess[i]),
                 textcoords='offset points', xytext=(5, 5), fontsize=9)

ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax3.set_xlabel('Benchmark Weight (%)', fontsize=11)
ax3.set_ylabel('Selection Excess (%)', fontsize=11)
ax3.set_title('Selection: Weight vs Stock Picking Excess', fontsize=13, fontweight='bold')
plt.colorbar(scatter2, ax=ax3, label='Select Effect (%)')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  图1：各行业的 BHB 三效应分解，柱子越高贡献越大")
print("  图2：右上象限=超配好行业(正配置)，左下象限=低配差行业(也是正配置)")
print("  图3：上方=选股跑赢基准(正选股)，下方=选股跑输(负选股)")

---

## 5. 多期时间序列：归因效应的时序变化

### 🎯 为什么要看多期？

单期的归因结果可能是噪声。多期时间序列可以回答：
- 配置效应是否**持续为正**？（说明择时能力稳定）
- 选股效应是否**持续为正**？（说明选股能力稳定）
- 交互效应是否**波动很大**？（说明超额收益来源不稳定）

### 📐 模拟设定

- 20 只股票 × 5 行业 × 12 个月
- 每月重新计算 BHB 归因
- 追踪累计归因效应

In [ ]:
# ========== 多期归因模拟 ==========
np.random.seed(42)

N_PERIODS = 12
industries = ['Technology', 'Consumer', 'Finance', 'Healthcare', 'Industrial']
wb_industry = np.array([0.25, 0.20, 0.20, 0.20, 0.15])
n_per_ind = 4

# 存储每期归因结果
period_results = []

for t in range(N_PERIODS):
    # 构造股票级数据
    stocks = []
    for ind_idx, ind_name in enumerate(industries):
        for s in range(n_per_ind):
            stocks.append({
                'Stock': f'{ind_name[0]}{s+1}',
                'Industry': ind_name,
                'wb_stock': wb_industry[ind_idx] / n_per_ind
            })
    stock_df = pd.DataFrame(stocks)

    # 组合权重：基准 + 持续偏好（超配Tech/Consumer）+ 随机扰动
    base_tilt = np.where(stock_df['Industry'] == 'Technology', 0.3,
                np.where(stock_df['Industry'] == 'Consumer', 0.15,
                np.where(stock_df['Industry'] == 'Finance', -0.2,
                np.where(stock_df['Industry'] == 'Healthcare', -0.1, -0.15))))
    stock_df['wp_stock'] = stock_df['wb_stock'] * (1 + base_tilt + np.random.normal(0, 0.15, len(stock_df)))
    stock_df['wp_stock'] = stock_df['wp_stock'].clip(lower=0.005)
    stock_df['wp_stock'] = stock_df['wp_stock'] / stock_df['wp_stock'].sum()

    # 收益率：Tech 有选股alpha，其他行业随机
    alpha = np.where(stock_df['Industry'] == 'Technology', 0.008, 0.0)
    stock_df['rb_stock'] = np.random.uniform(0.005, 0.06, len(stock_df))
    stock_df['rp_stock'] = stock_df['rb_stock'] + alpha + np.random.normal(0.002, 0.015, len(stock_df))

    # 运行归因
    res = bhb_attribution(stock_df)
    res['period'] = t + 1
    period_results.append(res)

# 汇总
ts_df = pd.DataFrame([{
    'Period': r['period'],
    'Rp': r['Rp'] * 100,
    'Rb': r['Rb'] * 100,
    'Active': r['active_return'] * 100,
    'Allocation': r['total_alloc'] * 100,
    'Selection': r['total_select'] * 100,
    'Interaction': r['total_interact'] * 100,
} for r in period_results])

# 累计值
ts_df['Cum_Active'] = ts_df['Active'].cumsum()
ts_df['Cum_Alloc'] = ts_df['Allocation'].cumsum()
ts_df['Cum_Select'] = ts_df['Selection'].cumsum()
ts_df['Cum_Interact'] = ts_df['Interaction'].cumsum()

print("📊 多期归因结果（每期增量 %）：")
print(ts_df[['Period', 'Active', 'Allocation', 'Selection', 'Interaction']].to_string(index=False))
print(f"\n  12 期累计：")
print(f"    总超额   = {ts_df['Cum_Active'].iloc[-1]:.4f}%")
print(f"    配置累计 = {ts_df['Cum_Alloc'].iloc[-1]:.4f}%")
print(f"    选股累计 = {ts_df['Cum_Select'].iloc[-1]:.4f}%")
print(f"    交互累计 = {ts_df['Cum_Interact'].iloc[-1]:.4f}%")

In [ ]:
# ========== 可视化：多期归因时序 ==========
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

periods = ts_df['Period'].values

# --- 图1: 每期三效应堆叠柱状图 ---
ax1 = axes[0, 0]
ax1.bar(periods, ts_df['Allocation'], 0.6, label='Allocation', color='#3498db', alpha=0.85)
ax1.bar(periods, ts_df['Selection'], 0.6,
        bottom=ts_df['Allocation'], label='Selection', color='#2ecc71', alpha=0.85)
bottom_2 = ts_df['Allocation'] + ts_df['Selection']
ax1.bar(periods, ts_df['Interaction'], 0.6,
        bottom=bottom_2, label='Interaction', color='#e67e22', alpha=0.85)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Period', fontsize=11)
ax1.set_ylabel('Contribution (%)', fontsize=11)
ax1.set_title('Per-Period BHB Attribution', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 累计三效应折线图 ---
ax2 = axes[0, 1]
ax2.plot(periods, ts_df['Cum_Alloc'], 'o-', color='#3498db', linewidth=2, label='Cum Alloc')
ax2.plot(periods, ts_df['Cum_Select'], 's-', color='#2ecc71', linewidth=2, label='Cum Select')
ax2.plot(periods, ts_df['Cum_Interact'], '^-', color='#e67e22', linewidth=2, label='Cum Interact')
ax2.plot(periods, ts_df['Cum_Active'], 'k--', linewidth=2.5, label='Cum Active Return')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Period', fontsize=11)
ax2.set_ylabel('Cumulative (%)', fontsize=11)
ax2.set_title('Cumulative Attribution Over Time', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# --- 图3: 各效应贡献占比（每期） ---
ax3 = axes[1, 0]
total_abs = ts_df['Allocation'].abs() + ts_df['Selection'].abs() + ts_df['Interaction'].abs()
pct_alloc = ts_df['Allocation'].abs() / total_abs * 100
pct_select = ts_df['Selection'].abs() / total_abs * 100
pct_interact = ts_df['Interaction'].abs() / total_abs * 100

ax3.stackplot(periods, pct_alloc, pct_select, pct_interact,
              labels=['Allocation', 'Selection', 'Interaction'],
              colors=['#3498db', '#2ecc71', '#e67e22'], alpha=0.85)
ax3.set_xlabel('Period', fontsize=11)
ax3.set_ylabel('Share of |Effect| (%)', fontsize=11)
ax3.set_title('Relative Importance of Each Effect', fontsize=13, fontweight='bold')
ax3.legend(fontsize=9, loc='upper right')
ax3.set_ylim(0, 100)
ax3.grid(alpha=0.3)

# --- 图4: 交互效应 vs 总超额散点图 ---
ax4 = axes[1, 1]
colors_dot = ['#2ecc71' if a > 0 else '#e74c3c' for a in ts_df['Active']]
ax4.scatter(ts_df['Interaction'], ts_df['Active'], s=100, c=colors_dot,
            edgecolors='black', zorder=5)
for i, p in enumerate(periods):
    ax4.annotate(f'P{p}', (ts_df['Interaction'].iloc[i], ts_df['Active'].iloc[i]),
                 textcoords='offset points', xytext=(4, 4), fontsize=8)

# 趋势线
z = np.polyfit(ts_df['Interaction'], ts_df['Active'], 1)
x_line = np.linspace(ts_df['Interaction'].min(), ts_df['Interaction'].max(), 100)
ax4.plot(x_line, np.poly1d(z)(x_line), 'r--', linewidth=2, alpha=0.7)
corr_val = ts_df['Interaction'].corr(ts_df['Active'])

ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax4.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax4.set_xlabel('Interaction Effect (%)', fontsize=11)
ax4.set_ylabel('Active Return (%)', fontsize=11)
ax4.set_title(f'Interaction vs Active Return (corr={corr_val:.2f})', fontsize=13, fontweight='bold')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  图1：每期的三效应堆叠柱状图，观察各效应的正负和大小变化")
print("  图2：累计归因折线图，黑色虚线=总超额，三条彩色线=各效应累计")
print("  图3：各效应占绝对值之和的比例，观察哪个效应是主要驱动")
print("  图4：交互效应 vs 总超额的相关性，高相关说明超额收益来源不稳定")

In [ ]:
# ========== 统计分析：各效应的稳定性 ==========
print("📊 各归因效应的统计特征")
print("═" * 60)

effects = ['Allocation', 'Selection', 'Interaction']
colors_eff = ['#3498db', '#2ecc71', '#e67e22']

for eff in effects:
    vals = ts_df[eff].values
    mean_v = vals.mean()
    std_v = vals.std(ddof=1)
    t_stat = mean_v / (std_v / np.sqrt(N_PERIODS))
    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=N_PERIODS - 1))
    hit_rate = (vals > 0).sum() / N_PERIODS * 100

    print(f"\n  {eff}:")
    print(f"    均值     = {mean_v:+.4f}%")
    print(f"    标准差   = {std_v:.4f}%")
    print(f"    T 统计量 = {t_stat:.4f}")
    print(f"    P 值     = {p_val:.6f}")
    print(f"    胜率     = {hit_rate:.0f}% ({(vals > 0).sum()}/{N_PERIODS})")
    if p_val < 0.05:
        print(f"    ✅ 在 5% 水平下显著不为零")
    else:
        print(f"    ⚠️ 在 5% 水平下不显著")

# scipy 验证
print(f"\n🔬 scipy.stats.ttest_1samp 验证：")
for eff in effects:
    t_s, p_s = stats.ttest_1samp(ts_df[eff].values, 0)
    print(f"  {eff}: t = {t_s:.4f}, p = {p_s:.6f}")

---

## 📌 核心概念回顾

### 📌 BHB 模型 (Brinson-Hood-Beebower Model)
- **定义**: 将投资组合超额收益分解为配置效应、选股效应和交互效应的归因模型
- **公式**: $R_p - R_b = \text{Alloc} + \text{Select} + \text{Interact}$
- **意义**: 量化基金经理的收益来源，区分择时能力和选股能力

### 📌 配置效应 (Allocation Effect)
- **定义**: 因为超配/低配了某些行业而产生的超额收益
- **公式**: $\sum_i (w_p^i - w_b^i)(r_b^i - R_b)$
- **含义**: 反映择时/资产配置能力
- **判断**: 超配了表现好的行业 → 正效应

### 📌 选股效应 (Selection Effect)
- **定义**: 在每个行业内，因为选了更好的股票而产生的超额收益
- **公式**: $\sum_i w_b^i(r_p^i - r_b^i)$
- **含义**: 反映选股能力
- **判断**: 行业内选股跑赢基准 → 正效应

### 📌 交互效应 (Interaction Effect)
- **定义**: 配置方向和选股能力的交叉贡献
- **公式**: $\sum_i (w_p^i - w_b^i)(r_p^i - r_b^i)$
- **含义**: 超配且选股好 → 正贡献；低配且选股差 → 也是正贡献（负负得正）
- **判断**: 交互效应大说明超额收益来源不稳定

### 🔗 完整流程
```
数据准备（组合权重、基准权重、收益率）
    ↓
按行业汇总（行业级权重、行业级加权收益）
    ↓
计算基准总收益 Rb
    ↓
BHB 三效应分解（配置 + 选股 + 交互）
    ↓
验证：三者之和 = 总超额收益
    ↓
多期时间序列 → 累计归因 → 稳定性分析
```

### 📝 BHB 三效应对比

| 效应 | 公式 | 反映能力 | 正值含义 |
|------|------|----------|----------|
| 配置 | $\sum (w_p^i - w_b^i)(r_b^i - R_b)$ | 择时/配置 | 超配了好行业 |
| 选股 | $\sum w_b^i(r_p^i - r_b^i)$ | 选股 | 行业内选股跑赢 |
| 交互 | $\sum (w_p^i - w_b^i)(r_p^i - r_b^i)$ | 交叉 | 配置方向与选股同向 |

### ❌ 常见误区

#### 误区 1: 把交互效应归功于"能力"
**✓ 正确做法**: 交互效应是配置方向和选股结果的"交叉项"，不代表独立的能力。如果交互效应占比很大，说明超额收益的来源不稳定——它依赖于"恰好在多配的行业里选了更好的股票"，这种好运不可持续。

#### 误区 2: 只看单期归因就下结论
**✓ 正确做法**: 单期归因可能是噪声。必须看多期时间序列，检验各效应是否持续、稳定。T 检验和胜率是判断标准。

#### 误区 3: 混淆"选股效应"和"选股能力"
**✓ 正确做法**: BHB 的选股效应 $\sum w_b^i(r_p^i - r_b^i)$ 用基准权重加权，衡量的是"如果保持基准配置，仅靠选股能赚多少"。它不等于基金经理的选股能力——因为实际组合的行业权重与基准不同。

#### 误区 4: 忽略 BHB 模型的假设
**✓ 正确做法**: BHB 模型假设收益率可以按行业线性分解，且不考虑交易成本、现金拖累等因素。在实际应用中，Brinson-Fachler 模型（引入基准总收益修正项）是更常用的变体。

#### 误区 5: 用 BHB 归因预测未来表现
**✓ 正确做法**: 归因分析是**事后**工具，用于解释过去的收益来源，而非预测未来。过去的配置效应为正不代表未来择时仍然有效。归因结果应与因子分析、风险模型结合使用。